# PTCG ABC — Colab 実行環境

大量対戦を Colab の CPU で回す。Mac 側はコード修正と局面確認に使う。

前提は `docs/design.md` の 0 節と 7 節。要点だけ再掲する。

- エンジンは `libcg.so` (Linux x86_64) を同梱しているので Colab で動く。Kaggle 本番も Linux
- 乱数シードは指定できない。再現性は対戦ログで担保する
- GPU は使わない。CUDA を使う処理がないため、割り当てても速くならない
- 資源は保証されないので、小分けに回して 1 戦ごとに Drive へ追記する

## 1. 環境の確認

In [ ]:
import os, platform, subprocess
print(platform.platform())
print('python', platform.python_version())
try:
    cpu = len(os.sched_getaffinity(0))
except AttributeError:
    cpu = os.cpu_count()
print('使える論理 CPU:', cpu)
print(subprocess.run(['free','-h'], capture_output=True, text=True).stdout)

## 2. インストール

Mac 側と同じ 1.32.2 に固定する。エンジンの挙動が版で変わると比較にならない。

In [ ]:
!pip -q install 'kaggle-environments==1.32.2'

## 3. リポジトリの取得

private リポジトリなので認証が要る。Colab の「シークレット」に `GITHUB_TOKEN` を
登録しておくと、下のセルがそれを使う。登録がなければ Drive に置いた zip を探す。

In [ ]:
import os, subprocess, glob
REPO = 'Tomomon2525/mutsugi_team'
BRANCH = 'tkawamura'
WORK = '/content/ptcg-abc'

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception as e:
    print('シークレット未設定:', e)

if os.path.isdir(os.path.join(WORK, '.git')):
    print(subprocess.run(['git','-C',WORK,'pull','--ff-only'], capture_output=True, text=True).stdout)
elif token:
    url = f'https://{token}@github.com/{REPO}.git'
    r = subprocess.run(['git','clone','--depth','50','-b',BRANCH,url,WORK], capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    from google.colab import drive
    drive.mount('/content/drive')
    zips = sorted(glob.glob('/content/drive/MyDrive/**/ptcg-abc*.zip', recursive=True))
    assert zips, 'GITHUB_TOKEN も Drive 上の zip も見つからない'
    print('展開:', zips[-1])
    !unzip -q -o "{zips[-1]}" -d /content/

os.chdir(WORK)
print(subprocess.run(['git','log','--oneline','-1'], capture_output=True, text=True).stdout)

## 4. エンジンが Linux で動くかの確認

In [ ]:
import sys, os
sys.path.insert(0, '/content/ptcg-abc/shared')
import ptcg
from kaggle_environments.envs.cabt.cg import sim
print('ロードした共有ライブラリ:', sim.lib._name)
print('カード種数:', len(ptcg.cards()))
print('id=648 ->', ptcg.name(648))

## 5. スループットの実測

判断基準は 1 戦あたりの速度ではなく **1 時間あたりの完了対戦数**。
論理 CPU 数を超えて並列を増やすと遅くなるので、そこも含めて測る。
Mac (8 論理 CPU, 並列 6) の数字と比べて、どちらで回すかを決める。

In [ ]:
%env PTCG_TIME_POOL=70
%env PTCG_MAX_SLICE=0.58
%env PTCG_RESERVE=5.2
%env PTCG_MIN_SLICE=0.03
!python tools/bench.py -j 1,2,4 -n 4

## 6. Drive を結果の保存先にする

ランタイム内だけに置くと切断で消える。1 戦ごとに追記されるので、
落ちても同じコマンドを流し直せば未完了分から再開する。

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
OUT = '/content/drive/MyDrive/ptcg-abc/league'
os.makedirs(OUT, exist_ok=True)
print(OUT)

## 7. 対戦を回す

並列数は 5 節の結果に合わせる。`-n` は総対戦数で、途中で切れても同じ数を
指定して流し直せばよい。

In [ ]:
JOBS = 2  # 5 節の最良値に合わせる
!python tools/league.py agents/d0_grimmsnarl champions/572_6/agent \
    -n 200 -j {JOBS} -b 25 -o {OUT}/ctx_vs_champ.jsonl

### 対面別に測る場合

In [ ]:
for foe in ['e_lucario', 'e_alakazam', 'e_dragapult']:
    print('=' * 60, foe)
    !python tools/league.py agents/d0_grimmsnarl agents/{foe} \
        -n 100 -j {JOBS} -b 25 -o {OUT}/d0_vs_{foe}.jsonl

## 8. 集計だけ見る

対戦を回さずに、保存済みの結果から現在の勝率を出す。

In [ ]:
import glob, json
for path in sorted(glob.glob(f'{OUT}/*.jsonl')):
    w = l = d = 0
    for line in open(path):
        try:
            r = json.loads(line)
        except Exception:
            continue
        if 'result' not in r:
            continue
        w += r['result'] > 0; l += r['result'] < 0; d += r['result'] == 0
    n = w + l + d
    if not n:
        continue
    z = (w / n - 0.5) / (0.25 / n) ** 0.5
    print(f"{os.path.basename(path):<32} {n:>4}戦 {w:>3}勝{l:>3}敗{d:>3}分  {w/n:>6.1%}  z={z:+.2f}")